In [3]:
pip install fake_useragent

     ------------------------------------ 201.1/201.1 kB 554.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
"""
Job Scraper for EU-based Jobs

This script scrapes job listings from specified company career pages, searching for roles 
that match a predefined list of keywords. It excludes unpaid internships and traineeships, 
ensuring only relevant positions are stored.

Key Features:
- Extracts job title and application link.
- Filters out unpaid internships/traineeships.
- Focuses only on EU-based job offers.
- Saves results to an Excel file, adding a new sheet for each run.

Requirements:
- Install dependencies: requests, BeautifulSoup, pandas, openpyxl
- Provide a list of career page URLs.
- Ensure access permissions to avoid 403 errors.

"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime
from fake_useragent import UserAgent

# List of career pages to scrape
career_pages = [
    
    # Add more URLs as needed
]

# Keywords for filtering jobs
keywords = [
    # add key words
]

# Function to check if the job is relevant
def is_relevant_job(title):
    title_lower = title.lower()
    return any(keyword.lower() in title_lower for keyword in keywords) and "unpaid" not in title_lower

# Function to scrape jobs from a career page
def scrape_jobs(url):
    try:
        response = requests.get(url, headers={"User-Agent": UserAgent().random})
        if response.status_code != 200:
            print(f"Error scraping {url}: {response.status_code}")
            return []

        soup = BeautifulSoup(response.text, "html.parser")
        jobs = []

        # Modify this based on how jobs are structured on each website
        for job in soup.find_all("a"):
            title = job.text.strip()
            link = job.get("href")

            if title and link and is_relevant_job(title):
                if not link.startswith("http"):  # Convert relative URLs to absolute
                    link = url.rstrip("/") + "/" + link.lstrip("/")
                jobs.append((title, link))

        return jobs
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return []

# Main script execution
all_jobs = []

for page in career_pages:
    jobs = scrape_jobs(page)
    all_jobs.extend(jobs)

# Save results in an Excel file
if all_jobs:
    filename = "job_listings.xlsx"
    sheet_name = f"Run_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')}"

    df = pd.DataFrame(all_jobs, columns=["Job Role", "Link"])

    try:
        with pd.ExcelWriter(filename, mode="a", if_sheet_exists="new") as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    except FileNotFoundError:
        with pd.ExcelWriter(filename, mode="w") as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Scraped {len(all_jobs)} jobs. Results saved to {filename}, sheet: {sheet_name}.")
else:
    print("No relevant jobs found.")


Error scraping https://corporate.coopculture.it/it/info/jobs/: 403
Error scraping https://job.ales-spa.com/: HTTPSConnectionPool(host='job.ales-spa.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLError(1, '[SSL: DH_KEY_TOO_SMALL] dh key too small (_ssl.c:997)')))
Error scraping https://www.revolut.com/working-at-revolut/: 403
Error scraping https://www.venetiancluster.eu/en/why-join-the-cluster/: 404
Scraped 312 jobs. Results saved to job_listings.xlsx, sheet: Run_2025-02-22_01-28.
